# Módulo 5 — Conexão com Snowflake

Neste notebook vamos entender a classe de conexão usada no projeto de qualidade de dados
da Equatorial e aprender a usá-la para executar queries e carregar dados em DataFrames.

## O que vamos ver

1. Estrutura do módulo `snowflake_db.py`
2. Autenticação por chave privada (PKI) — padrão corporativo
3. Configuração via variáveis de ambiente (`.env`)
4. Como usar a classe `SnowflakeDB` com context manager
5. Funções utilitárias: `read_sql_to_dataframe` e `write_dataframe_to_snowflake`

---

> **Credenciais Seguras:** 
> - A sua chave privada deve estar em `.secrets/snowflake.p8` na raiz do projeto.
> - As configurações de conta, usuário e senha da chave devem estar no arquivo `.env` na raiz.
>
> **Segurança:** O arquivo `.env` e a pasta `.secrets/` já estão no `.gitignore` para sua proteção.

## 1. Verificando dependências

In [10]:
import importlib

dependencias = {
    "snowflake.connector": "snowflake-connector-python",
    "cryptography": "cryptography",
    "jinja2": "jinja2",
    "pandas": "pandas",
}

todas_ok = True
for modulo, pacote in dependencias.items():
    try:
        importlib.import_module(modulo)
        print(f"  OK  {modulo}")
    except ImportError:
        print(f"  ERRO {modulo} — instale com: uv add {pacote}")
        todas_ok = False

print()
print("Tudo pronto!" if todas_ok else "Instale as dependências acima antes de continuar.")

  OK  snowflake.connector
  OK  cryptography
  OK  jinja2
  OK  pandas

Tudo pronto!


## 2. Importando o módulo do projeto

In [11]:
import sys
import os
from pathlib import Path
from dotenv import load_dotenv

# --- Localizar snowflake_db.py independente do CWD do Jupyter ---
cwd = Path().resolve()
print(f"CWD detectado: {cwd}")

candidatos = [
    cwd,                                              # CWD já é modulo5/
    cwd / "modulo5_snowflake_qualidade",              # CWD é raiz do projeto
    cwd.parent / "modulo5_snowflake_qualidade",       # CWD é outro módulo
    Path(__file__).parent if "__file__" in dir() else cwd,  # via __file__
]

modulo5_dir = next(
    (p for p in candidatos if (p / "snowflake_db.py").exists()),
    None,
)

if modulo5_dir is None:
    raise FileNotFoundError(
        f"snowflake_db.py não encontrado.\n"
        f"CWD={cwd}\n"
        f"Caminhos testados:\n" + "\n".join(f"  {p}" for p in candidatos)
    )

print(f"snowflake_db.py encontrado em: {modulo5_dir}")

# --- Adicionar ao sys.path ---
if str(modulo5_dir) not in sys.path:
    sys.path.insert(0, str(modulo5_dir))

# --- Carregar .env (procura na raiz do projeto) ---
for env_candidate in [cwd / ".env", cwd.parent / ".env", modulo5_dir.parent / ".env"]:
    if env_candidate.exists():
        load_dotenv(dotenv_path=env_candidate, override=True)
        print(f".env carregado: {env_candidate}")
        break

# --- Importar o módulo ---
from snowflake_db import (
    SnowflakeDB,
    SNOWFLAKE_CONFIG,
    read_sql_file,
    read_sql_to_dataframe,
    write_dataframe_to_snowflake,
)

import pandas as pd

print(f"\nMódulo importado com sucesso.")
print(f"Conta  : {SNOWFLAKE_CONFIG['account']}")
print(f"Usuário: {SNOWFLAKE_CONFIG['user']}")

CWD detectado: /home/vinicius/projects/t2t/python-learning
snowflake_db.py encontrado em: /home/vinicius/projects/t2t/python-learning/modulo5_snowflake_qualidade
.env carregado: /home/vinicius/projects/t2t/python-learning/.env

Módulo importado com sucesso.
Conta  : khb56279.us-east-1
Usuário: STECH2THINK_SNOWFLAKE@GRUPOEQUATORIALENERGIA.ONMICROSOFT.COM


## 3. Entendendo a autenticação por chave privada

A Equatorial usa **PKI (Public Key Infrastructure)** ao invés de senha para autenticar no Snowflake.
Isso é mais seguro porque:

- A chave privada nunca trafega pela rede
- Cada usuário/serviço tem sua própria chave
- É possível revogar acessos sem mudar senhas

```
Arquivo: rsa_key.p8  (chave privada — NUNCA compartilhe)
   └── Protegida por passphrase
   └── O módulo lê, descriptografa e converte para DER/PKCS8
   └── O conector Snowflake usa os bytes resultantes
```

O fluxo interno da classe `SnowflakeDB`:

In [12]:
# Inspecionando a estrutura da classe sem executar a conexão
import inspect

# Listar os métodos públicos da classe
metodos = [
    (nome, m)
    for nome, m in inspect.getmembers(SnowflakeDB, predicate=inspect.isfunction)
    if not nome.startswith("_")
]

print("Métodos públicos de SnowflakeDB:\n")
for nome, metodo in metodos:
    sig = inspect.signature(metodo)
    doc = (metodo.__doc__ or "").strip().split("\n")[0]
    print(f"  .{nome}{sig}")
    print(f"     → {doc}")
    print()

Métodos públicos de SnowflakeDB:

  .execute_query(self, *args, **kwargs)
     → 

  .query_to_dataframe(self, *args, **kwargs)
     → 

  .write_dataframe(self, *args, **kwargs)
     → 



## 4. Conectando e executando uma query

O padrão recomendado é o **context manager** (`with`).  
Ele garante que a conexão seja fechada mesmo se ocorrer um erro.

```python
with SnowflakeDB(**SNOWFLAKE_CONFIG) as db:
    df = db.query_to_dataframe("SELECT ...")
# conexão já está fechada aqui
```

In [13]:
# Teste de conexão — verifique o contexto da sessão
try:
    with SnowflakeDB(**SNOWFLAKE_CONFIG) as db:
        df_ctx = db.query_to_dataframe(
            """
            SELECT
                CURRENT_USER()     AS usuario,
                CURRENT_ROLE()     AS role,
                CURRENT_DATABASE() AS database,
                CURRENT_SCHEMA()   AS schema,
                CURRENT_WAREHOUSE() AS warehouse,
                CURRENT_VERSION()  AS versao_snowflake
            """
        )

    print("Conexão bem-sucedida!\n")
    print(df_ctx.T.to_string(header=False))  # exibe na vertical

except Exception as e:
    print(f"Erro: {type(e).__name__}: {e}")
    print("\nVerifique:")
    print("  1. Se o arquivo snowflake.p8 está em .secrets/snowflake.p8")
    print("  2. Se a passphrase e o usuário no arquivo .env estão corretos")
    print("  3. Sua conexão com a internet")

INFO: Conectando ao Snowflake (account=khb56279.us-east-1, database=EQTLINFO_HML, schema=EQTL_MA)...
INFO: Snowflake Connector for Python Version: 4.3.0, Python Version: 3.12.3, Platform: Linux-6.17.0-19-generic-x86_64-with-glibc2.39
INFO: Connecting to GLOBAL Snowflake domain
INFO: Conexão estabelecida com sucesso.
INFO: Conexão encerrada.


Conexão bem-sucedida!

USUARIO           STECH2THINK_SNOWFLAKE@GRUPOEQUATORIALENERGIA.ONMICROSOFT.COM
ROLE                                                                    PUBLIC
DATABASE                                                          EQTLINFO_HML
SCHEMA                                                                 EQTL_MA
WAREHOUSE                                                          WH_EQTLINFO
VERSAO_SNOWFLAKE                                                        10.9.2


## 5. Usando `read_sql_to_dataframe`

Para análises do dia a dia, use a função utilitária — ela abre e fecha a conexão automaticamente.

In [14]:
# Query direta como string
df_classes = read_sql_to_dataframe(
    """
    SELECT
        TAB_CADASTRO.INSTALACAO AS CR_NUMERO,
        CONSUMIDOR.INSTALACAO_ID AS INSTALACAO_ID,
        TRY_CAST(TAB_CADASTRO.CONTRATO AS INTEGER) AS CONTRATO,
        TRY_CAST(TAB_CADASTRO.CONTA_CONTRATO AS INTEGER) AS CONTA_CONTRATO,
        TRY_CAST(TAB_CADASTRO.MEDIDOR AS INTEGER) AS MEDIDOR_ID,
        TAB_CADASTRO.TIPO_LOCAL_CONSUMO,
        TAB_CADASTRO.GENERO,
        TAB_CADASTRO.DATA_NASCIMENTO,
        TAB_CADASTRO.ESTADO_CIVIL,
        TAB_CADASTRO.LATITUDE,
        TAB_CADASTRO.LONGITUDE,
        TAB_CADASTRO.DATA_INSTALACAO_MEDIDOR
    FROM (
            SELECT *
            FROM EQTLINFO_PRD.EQTL_PA.TAB_CADASTRO
            ORDER BY RANDOM()
            LIMIT 10
    ) TAB_CADASTRO
    LEFT JOIN EQTLINFO_RAW.OPER_PA.CONSUMIDOR CONSUMIDOR 
    ON TAB_CADASTRO.INSTALACAO = LPAD(
            REGEXP_REPLACE(TO_VARCHAR(CONSUMIDOR.CR_NUMERO), '[^0-9]', ''),
            LENGTH(TAB_CADASTRO.INSTALACAO),
            '0'
        )
    """
)

print(f"Linhas retornadas: {len(df_classes)}")
df_classes

INFO: Conectando ao Snowflake (account=khb56279.us-east-1, database=EQTLINFO_HML, schema=EQTL_MA)...
INFO: Snowflake Connector for Python Version: 4.3.0, Python Version: 3.12.3, Platform: Linux-6.17.0-19-generic-x86_64-with-glibc2.39
INFO: Connecting to GLOBAL Snowflake domain
INFO: Conexão estabelecida com sucesso.
INFO: Conexão encerrada.


Linhas retornadas: 10


,CR_NUMERO,INSTALACAO_ID,CONTRATO,CONTA_CONTRATO,MEDIDOR_ID,TIPO_LOCAL_CONSUMO,GENERO,DATA_NASCIMENTO,ESTADO_CIVIL,LATITUDE,LONGITUDE,DATA_INSTALACAO_MEDIDOR
0,0020203153,None,20203153.0,2.020315e+07,1.101051e+10,Casa,Masculino,1963-12-05,Solteiro(a),-1.46749006667,-48.45869820417,2017-07-17
1,2000745780,None,NaN,NaN,NaN,Casa,None,None,None,-1.400726287685,-48.461638242175,None
2,0014515232,None,14515232.0,1.451523e+07,2.198697e+06,Indefinido,Masculino,1975-05-24,Solteiro(a),-1.4223502,-48.4671125,2015-01-22
3,0080895313,None,80895313.0,8.089531e+07,NaN,Casa,Indefinido,1942-06-05,None,-1.406058084344,-51.646271611373,None
4,0007797850,None,7797850.0,7.797850e+06,1.101039e+10,Casa,Masculino,1990-01-01,Divorciado(a),-1.35158478333,-48.4148806,2017-07-11
5,0013686882,None,NaN,NaN,NaN,Casa,None,None,None,-1.349198063103,-48.476250828930,None
6,0099932511,None,505473531.0,3.027610e+09,1.349991e+07,Casa,Feminino,1999-10-07,Outros,-5.379010593333,-49.124679420000,2015-01-28
7,2001192236,2204163.00,505632651.0,3.028856e+09,2.401061e+10,Casa,Indefinido,1997-04-17,Solteiro(a),-2.453536279500,-54.744262341410,2023-12-06
8,2001241761,2085379.00,505765121.0,3.029898e+09,1.406067e+10,Casa,Indefinido,1986-06-28,None,-0.897620770000,-47.012805680000,2024-04-13
9,0011224130,None,503494017.0,3.012003e+09,1.101235e+10,Casa,Masculino,1982-03-05,Casado(a),-1.45804789,-48.45349836,2020-06-03


## 6. Queries a partir de arquivos `.sql` com templates Jinja2

Para queries mais complexas, o ideal é salvá-las em arquivos `.sql` separados.
O módulo suporta **templates Jinja2** para passar parâmetros dinamicamente.

In [19]:
# Diretório do projeto
project_root = Path().resolve()

# Query parametrizada por CR_NUMERO — lendo de arquivo .sql externo
sql_consumidor = project_root / "queries" / "tb_consumidor_por_cr.sql"

# Definir o CR_NUMERO que queremos consultar
cr_numero = "0002997827"

# read_sql_file processa o template Jinja2 e substitui {{ cr_numero }}
query_renderizada = read_sql_file(sql_consumidor, cr_numero=cr_numero)

print(f"Buscando consumidor: CR_NUMERO = {cr_numero}")
print(f"Arquivo SQL         : {sql_consumidor.name}")
print()
print("Query renderizada:")
print(query_renderizada)

Buscando consumidor: CR_NUMERO = 0002997827
Arquivo SQL         : tb_consumidor_por_cr.sql

Query renderizada:
SELECT
  TAB_CADASTRO.INSTALACAO AS CR_NUMERO,
  CONSUMIDOR.INSTALACAO_ID AS INSTALACAO_ID,
  TRY_CAST(TAB_CADASTRO.CONTRATO AS INTEGER) AS CONTRATO,
  TRY_CAST(TAB_CADASTRO.CONTA_CONTRATO AS INTEGER) AS CONTA_CONTRATO,
  TRY_CAST(TAB_CADASTRO.MEDIDOR AS INTEGER) AS MEDIDOR_ID,
  TAB_CADASTRO.TIPO_LOCAL_CONSUMO,
  TAB_CADASTRO.GENERO,
  TAB_CADASTRO.DATA_NASCIMENTO,
  TAB_CADASTRO.ESTADO_CIVIL,
  TAB_CADASTRO.LATITUDE,
  TAB_CADASTRO.LONGITUDE,
  TAB_CADASTRO.DATA_INSTALACAO_MEDIDOR
FROM EQTLINFO_PRD.EQTL_PA.TAB_CADASTRO TAB_CADASTRO
LEFT JOIN EQTLINFO_RAW.OPER_PA.CONSUMIDOR CONSUMIDOR
  ON TAB_CADASTRO.INSTALACAO = LPAD(
    REGEXP_REPLACE(TO_VARCHAR(CONSUMIDOR.CR_NUMERO), '[^0-9]', ''),
    LENGTH(TAB_CADASTRO.INSTALACAO),
    '0'
  )
WHERE TAB_CADASTRO.INSTALACAO = '0002997827'


In [20]:
# Executar a query e trazer o cadastro completo do consumidor
df_consumidor = read_sql_to_dataframe(sql_consumidor, cr_numero=cr_numero)

print(f"Registros encontrados: {len(df_consumidor)}")
df_consumidor.T  # exibe na vertical para facilitar a leitura

INFO: Conectando ao Snowflake (account=khb56279.us-east-1, database=EQTLINFO_HML, schema=EQTL_MA)...
INFO: Snowflake Connector for Python Version: 4.3.0, Python Version: 3.12.3, Platform: Linux-6.17.0-19-generic-x86_64-with-glibc2.39
INFO: Connecting to GLOBAL Snowflake domain
INFO: Conexão estabelecida com sucesso.
INFO: Conexão encerrada.


Registros encontrados: 1


,0
CR_NUMERO,0002997827
INSTALACAO_ID,None
CONTRATO,2997827
CONTA_CONTRATO,2997827
MEDIDOR_ID,1304186506
TIPO_LOCAL_CONSUMO,Casa
GENERO,Feminino
DATA_NASCIMENTO,1967-03-07
ESTADO_CIVIL,Solteiro(a)
LATITUDE,-1.3465397


## 7. Queries parametrizadas (seguras contra SQL injection)

Para valores dinâmicos dentro da query, use o mecanismo de parâmetros `%s` do conector
ao invés de f-strings. Isso previne **SQL injection**.

In [ ]:
# ✅ CORRETO — parâmetro passado via template Jinja2, nunca via f-string
# Trocar o CR_NUMERO é só mudar a variável abaixo

crs_para_consultar = ["0002997827", "0012514654", "2001192287"]

resultados = []
for cr in crs_para_consultar:
    df_cr = read_sql_to_dataframe(sql_consumidor, cr_numero=cr)
    if not df_cr.empty:
        resultados.append(df_cr)
        print(f"  CR {cr} → encontrado | medidor={df_cr['MEDIDOR_ID'].iloc[0]}")
    else:
        print(f"  CR {cr} → não encontrado")

df_multiplos = pd.concat(resultados, ignore_index=True) if resultados else pd.DataFrame()
print(f"\nTotal de registros retornados: {len(df_multiplos)}")
df_multiplos[["CR_NUMERO", "CONTRATO", "MEDIDOR_ID", "GENERO", "TIPO_LOCAL_CONSUMO"]]

In [ ]:
from datetime import datetime

# Enriquecer df_multiplos com colunas calculadas antes de gravar
df_para_gravar = df_multiplos.copy()

# Coluna de qualidade: considera campos obrigatórios preenchidos
df_para_gravar["SCORE_QUALIDADE"] = (
    df_para_gravar[["CONTRATO", "MEDIDOR_ID", "LATITUDE", "LONGITUDE", "DATA_NASCIMENTO"]]
    .notna()
    .mean(axis=1)
    .mul(100)
    .round(1)
)

# Flag simples: cadastro completo se todos os 5 campos estão preenchidos
df_para_gravar["FLG_CADASTRO_COMPLETO"] = (
    df_para_gravar["SCORE_QUALIDADE"] == 100.0
)

# Carimbo de processamento
df_para_gravar["DAT_PROCESSAMENTO"] = datetime.now()

print("Prévia do DataFrame a ser gravado:")
print(
    df_para_gravar[[
        "CR_NUMERO", "CONTRATO", "MEDIDOR_ID",
        "SCORE_QUALIDADE", "FLG_CADASTRO_COMPLETO", "DAT_PROCESSAMENTO"
    ]].to_string(index=False)
)

# Para gravar de fato no Snowflake, descomente as linhas abaixo:
# write_dataframe_to_snowflake(
#     df_para_gravar,
#     table_name="TB_QUALIDADE_CADASTRO_CR",
#     auto_create_table=True,
#     overwrite=False,
# )
# print(f"\n{len(df_para_gravar)} registro(s) gravado(s) com sucesso!")